# Nanochat D8 Quant Run on Kaggle

This notebook starts from the saved `d8_kaggle` SFT checkpoint and exports quantized artifacts:

- `int8_linear` with default embedding behavior
- `fp16_copy`
- AWQ-style `awq_int4`

Recommended Kaggle setup:

- GPU enabled (`2x Tesla T4` available, one GPU is enough for serving/eval)
- Internet enabled
- Attach `nanochat-codes`
- Attach `nanochat-d8-sft-cache`

The notebook imports only the tokenizer and `chatsft_checkpoints/d8_kaggle` from the SFT cache, then writes quant artifacts under `chatquant_exports`.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

INPUT_ROOT = Path('/kaggle/input')

CODE_INPUTS = sorted(INPUT_ROOT.glob('**/nanochat-codes'))
SFT_CACHE_INPUTS = sorted(INPUT_ROOT.glob('**/nanochat-d8-sft-cache'))

assert CODE_INPUTS, 'Attach the nanochat-codes Kaggle dataset'
assert SFT_CACHE_INPUTS, 'Attach the nanochat-d8-sft-cache Kaggle dataset'

CODE_INPUT = CODE_INPUTS[0]
SFT_CACHE_INPUT = SFT_CACHE_INPUTS[0]

WORK_REPO = Path('/kaggle/working/nanochat')
WORK_CACHE = Path('/kaggle/working/nanochat_cache')
HF_CACHE = Path('/kaggle/temp/huggingface_cache')

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)

required_cache_paths = [
    Path('tokenizer'),
    Path('chatsft_checkpoints') / 'd8_kaggle',
]
for rel_path in required_cache_paths:
    src_path = SFT_CACHE_INPUT / rel_path
    dst_path = WORK_CACHE / rel_path
    assert src_path.exists(), f'Missing required cache path in attached dataset: {src_path}'
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    if src_path.is_dir():
        shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dst_path)

os.environ['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE / 'hub')
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')

print('Code input:', CODE_INPUT)
print('SFT cache input:', SFT_CACHE_INPUT)
print('Working repo:', WORK_REPO)
print('Nanochat cache:', WORK_CACHE)
print('HF cache:', HF_CACHE)
subprocess.run(['df', '-h', '/kaggle/working', '/kaggle/temp'], check=False)
subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)


## Install Dependencies

Install the runtime packages used by quant export, eval, and the quantized web server.


In [ ]:
packages = [
    'datasets>=4.0.0',
    'fastapi>=0.115.0',
    'filelock>=3.0.0',
    'psutil>=7.1.0',
    'pydantic>=2.0.0',
    'requests>=2.32.0',
    'rustbpe>=0.1.0',
    'tiktoken>=0.11.0',
    'tokenizers>=0.22.0',
    'transformers>=4.57.3',
    'uvicorn>=0.30.0',
    'zstandard>=0.25.0',
]
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '--upgrade-strategy',
    'only-if-needed',
] + packages)
print('Dependencies installed')


## Configure Runtime

Keep compilation disabled and use float32 while exporting/evaluating quantized artifacts.


In [ ]:
env = os.environ.copy()
env['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
env['HF_HOME'] = str(HF_CACHE)
env['HF_HUB_CACHE'] = str(HF_CACHE / 'hub')
env['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')
env['HF_HUB_DISABLE_XET'] = '1'
env['HF_HUB_DOWNLOAD_TIMEOUT'] = '600'
env['HF_HUB_ETAG_TIMEOUT'] = '60'
env['TMPDIR'] = '/kaggle/temp'
env['PYTHONUNBUFFERED'] = '1'
env['NANOCHAT_DTYPE'] = 'float32'
env['NANOCHAT_DISABLE_COMPILE'] = '1'
env['TORCH_COMPILE_DISABLE'] = '1'
env['OMP_NUM_THREADS'] = '1'

os.environ.update(env)

def run_cmd(cmd, **kwargs):
    cmd = [str(x) for x in cmd]
    print('\\n$', ' '.join(cmd))
    return subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True, **kwargs)

def latest_model_step(checkpoint_dir):
    files = sorted(Path(checkpoint_dir).glob('model_*.pt'))
    assert files, f'No model_*.pt files found in {checkpoint_dir}'
    return max(int(path.stem.split('_')[-1]) for path in files)

def latest_quant_artifact(export_dir):
    files = sorted(Path(export_dir).glob('quant_*.pt'))
    assert files, f'No quant_*.pt files found in {export_dir}'
    return files[-1]

print('NANOCHAT_BASE_DIR:', env['NANOCHAT_BASE_DIR'])
print('HF_HOME:', env['HF_HOME'])
print('NANOCHAT_DTYPE:', env['NANOCHAT_DTYPE'])


In [ ]:
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))

MODEL_TAG = 'd8_kaggle'
SFT_DIR = WORK_CACHE / 'chatsft_checkpoints' / MODEL_TAG
SFT_STEP = latest_model_step(SFT_DIR)
print('SFT checkpoint dir:', SFT_DIR)
print('Latest SFT step:', SFT_STEP)


## Validate Repo And SFT Cache

Check the active Kaggle working copy and attached SFT cache before exporting artifacts.


In [ ]:
quant_scripts = [
    'scripts/chat_quantize.py',
    'scripts/chat_quant_awq.py',
    'scripts/chat_quant_eval.py',
    'scripts/chat_web_quant.py',
]

for rel_path in quant_scripts:
    assert (WORK_REPO / rel_path).exists(), f'Missing {rel_path}'
assert (WORK_CACHE / 'tokenizer').exists(), 'Missing tokenizer directory'
assert SFT_DIR.exists(), f'Missing SFT checkpoint dir: {SFT_DIR}'
assert sorted(SFT_DIR.glob('model_*.pt')), f'No model_*.pt files found in {SFT_DIR}'
assert sorted(SFT_DIR.glob('meta_*.json')), f'No meta_*.json files found in {SFT_DIR}'

run_cmd([sys.executable, '-m', 'py_compile'] + quant_scripts)

for module in [
    'scripts.chat_quantize',
    'scripts.chat_quant_awq',
    'scripts.chat_quant_eval',
    'scripts.chat_web_quant',
]:
    result = subprocess.run(
        [sys.executable, '-m', module, '--help'],
        cwd=WORK_REPO,
        env=env,
        text=True,
        capture_output=True,
        check=True,
    )
    print('\\nHELP', module)
    print('\\n'.join(result.stdout.splitlines()[:18]))


## Export INT8 And FP16 Artifacts

Export `int8_linear` and `fp16_copy` artifacts from the SFT checkpoint.


In [ ]:
EXPORT_ROOT = WORK_CACHE / 'chatquant_exports'
INT8_LINEAR_DIR = EXPORT_ROOT / f'{MODEL_TAG}_int8_linear'
FP16_COPY_DIR = EXPORT_ROOT / f'{MODEL_TAG}_fp16_copy'
OVERWRITE_QUANT_EXPORTS = True

if OVERWRITE_QUANT_EXPORTS:
    for path in [INT8_LINEAR_DIR, FP16_COPY_DIR]:
        shutil.rmtree(path, ignore_errors=True)

for path in [INT8_LINEAR_DIR, FP16_COPY_DIR]:
    path.mkdir(parents=True, exist_ok=True)

run_cmd([
    sys.executable, '-m', 'scripts.chat_quantize',
    '--source', 'sft',
    '--model-tag', MODEL_TAG,
    '--step', str(SFT_STEP),
    '--method', 'int8_linear',
    '--output-dir', INT8_LINEAR_DIR,
])
run_cmd([
    sys.executable, '-m', 'scripts.chat_quantize',
    '--source', 'sft',
    '--model-tag', MODEL_TAG,
    '--step', str(SFT_STEP),
    '--method', 'fp16_copy',
    '--output-dir', FP16_COPY_DIR,
])

INT8_LINEAR_ARTIFACT = latest_quant_artifact(INT8_LINEAR_DIR)
FP16_COPY_ARTIFACT = latest_quant_artifact(FP16_COPY_DIR)
print('INT8 artifact:', INT8_LINEAR_ARTIFACT)
print('FP16 artifact:', FP16_COPY_ARTIFACT)


## Export AWQ INT4 Artifact

Export an AWQ-style INT4 artifact using GSM8K calibration prompts.


In [ ]:
RUN_AWQ_EXPORT = True
AWQ_CALIBRATION_EXAMPLES = 128
AWQ_MAX_CALIBRATION_TOKENS = 512
AWQ_ALPHA = 0.5
AWQ_INT4_DIR = EXPORT_ROOT / f'{MODEL_TAG}_awq_int4'

if RUN_AWQ_EXPORT:
    if OVERWRITE_QUANT_EXPORTS:
        shutil.rmtree(AWQ_INT4_DIR, ignore_errors=True)
    AWQ_INT4_DIR.mkdir(parents=True, exist_ok=True)
    run_cmd([
        sys.executable, '-m', 'scripts.chat_quant_awq',
        '--source', 'sft',
        '--model-tag', MODEL_TAG,
        '--step', str(SFT_STEP),
        '--calibration-source', 'gsm8k',
        '--calibration-examples', str(AWQ_CALIBRATION_EXAMPLES),
        '--max-calibration-tokens', str(AWQ_MAX_CALIBRATION_TOKENS),
        '--alpha', str(AWQ_ALPHA),
        '--output-dir', AWQ_INT4_DIR,
    ])
    AWQ_INT4_ARTIFACT = latest_quant_artifact(AWQ_INT4_DIR)
    print('AWQ artifact:', AWQ_INT4_ARTIFACT)
else:
    AWQ_INT4_ARTIFACT = None
    print('Skipping AWQ export')


## Inspect Quant Artifacts

Check artifact metadata and sizes before evaluation or saving.


In [ ]:
QUANT_ARTIFACTS = {
    'int8_linear': INT8_LINEAR_ARTIFACT,
    'fp16_copy': FP16_COPY_ARTIFACT,
}
if AWQ_INT4_ARTIFACT is not None:
    QUANT_ARTIFACTS['awq_int4'] = AWQ_INT4_ARTIFACT

for name, artifact_path in QUANT_ARTIFACTS.items():
    assert Path(artifact_path).exists(), f'Missing artifact: {artifact_path}'
    data = torch.load(artifact_path, map_location='cpu')
    print()
    print(name, artifact_path)
    print('quant_method:', data.get('quant_method'))
    print('source_model_tag:', data.get('source_model_tag'))
    print('source_step:', data.get('source_step'))
    print('artifact_format_version:', data.get('artifact_format_version'))
    print('size bytes:', Path(artifact_path).stat().st_size)

int8_data = torch.load(INT8_LINEAR_ARTIFACT, map_location='cpu')
fp16_data = torch.load(FP16_COPY_ARTIFACT, map_location='cpu')
assert int8_data['quant_method'] == 'int8_linear'
assert int8_data['quantize_embeddings'] is False
assert fp16_data['quant_method'] == 'fp16_copy'
assert fp16_data['quantize_embeddings'] is True
if AWQ_INT4_ARTIFACT is not None:
    awq_data = torch.load(AWQ_INT4_ARTIFACT, map_location='cpu')
    assert awq_data['quant_method'] == 'awq_int4'

subprocess.run(['du', '-sh', str(EXPORT_ROOT)], check=False)


## Quantized Evaluation

Evaluate the exported quant artifacts on a fixed GSM8K slice.


In [ ]:
RUN_QUANT_EVAL = True
EVAL_TASK_NAME = 'GSM8K'
EVAL_MAX_PROBLEMS = 100
EVAL_MAX_NEW_TOKENS = 256
EVAL_BATCH_SIZE = 2

if RUN_QUANT_EVAL:
    for name, artifact_path in QUANT_ARTIFACTS.items():
        print('\\nEvaluating', name, artifact_path)
        run_cmd([
            sys.executable, '-m', 'scripts.chat_quant_eval',
            '--source', 'sft',
            '--model-tag', MODEL_TAG,
            '--step', str(SFT_STEP),
            '--quant-artifact', artifact_path,
            '--task-name', EVAL_TASK_NAME,
            '--max-problems', str(EVAL_MAX_PROBLEMS),
            '--max-new-tokens', str(EVAL_MAX_NEW_TOKENS),
            '--num-samples', '1',
            '--batch-size', str(EVAL_BATCH_SIZE),
        ])
else:
    print('Skipping quantized eval')


## Runtime-Quantized Eval

Exercise the `--runtime-quantized` path on the INT8 artifact.


In [ ]:
RUN_RUNTIME_QUANTIZED_EVAL = True
RUNTIME_EVAL_MAX_PROBLEMS = 20

if RUN_RUNTIME_QUANTIZED_EVAL:
    run_cmd([
        sys.executable, '-m', 'scripts.chat_quant_eval',
        '--quant-artifact', INT8_LINEAR_ARTIFACT,
        '--runtime-quantized',
        '--task-name', 'GSM8K',
        '--max-problems', str(RUNTIME_EVAL_MAX_PROBLEMS),
        '--max-new-tokens', '128',
        '--num-samples', '1',
        '--batch-size', '1',
    ])
else:
    print('Skipping runtime-quantized eval')


## Output Manifest

Write a compact manifest of the artifacts produced by this notebook.


In [ ]:
manifest = {
    'model_tag': MODEL_TAG,
    'sft_checkpoint_dir': str(SFT_DIR),
    'sft_step': SFT_STEP,
    'export_root': str(EXPORT_ROOT),
    'quant_artifacts': {name: str(path) for name, path in QUANT_ARTIFACTS.items()},
    'artifact_sizes_bytes': {
        name: Path(path).stat().st_size
        for name, path in QUANT_ARTIFACTS.items()
        if Path(path).exists()
    },
    'tested_scripts': quant_scripts,
    'eval': {
        'task_name': EVAL_TASK_NAME,
        'max_problems': EVAL_MAX_PROBLEMS,
        'max_new_tokens': EVAL_MAX_NEW_TOKENS,
    },
}
manifest_path = Path('/kaggle/working/nanochat_d8_quant_manifest.json')
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(manifest_path)
print(manifest_path.read_text())

print('Artifact sizes:')
for name, path in QUANT_ARTIFACTS.items():
    print(name, path, Path(path).stat().st_size, 'bytes')
subprocess.run(['du', '-sh', str(EXPORT_ROOT)], check=False)


In [ ]:
# Optional: save the quant artifact cache as a Kaggle Dataset.
import json

QUANT_CACHE_DIR = Path('/kaggle/working/nanochat_d8_quant_cache')

DATASET_ID = 'yshuaiqin/nanochat-d8-quant-cache'
TITLE = 'nanochat d8 quant cache'
VERSION_MESSAGE = f'Save {MODEL_TAG} quant artifacts from SFT step {SFT_STEP}'
UPLOAD_ACTION = 'create'  # use 'version' after the Dataset already exists

required_paths = [EXPORT_ROOT, WORK_CACHE / 'tokenizer'] + [Path(path) for path in QUANT_ARTIFACTS.values()]
for required_path in required_paths:
    assert required_path.exists(), f'Missing required path: {required_path}'

if QUANT_CACHE_DIR.exists():
    shutil.rmtree(QUANT_CACHE_DIR)
QUANT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(EXPORT_ROOT, QUANT_CACHE_DIR / 'chatquant_exports')
shutil.copytree(WORK_CACHE / 'tokenizer', QUANT_CACHE_DIR / 'tokenizer')
if manifest_path.exists():
    shutil.copy2(manifest_path, QUANT_CACHE_DIR / manifest_path.name)

metadata = {
    'title': TITLE,
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}

metadata_path = QUANT_CACHE_DIR / 'dataset-metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2))

print('Folder size:')
subprocess.run(['du', '-sh', str(QUANT_CACHE_DIR)], check=False)

if UPLOAD_ACTION == 'create':
    cmd = [
        'kaggle', 'datasets', 'create',
        '-p', str(QUANT_CACHE_DIR),
        '--dir-mode', 'tar',
    ]
elif UPLOAD_ACTION == 'version':
    cmd = [
        'kaggle', 'datasets', 'version',
        '-p', str(QUANT_CACHE_DIR),
        '-m', VERSION_MESSAGE,
        '--dir-mode', 'tar',
    ]
else:
    raise ValueError("UPLOAD_ACTION must be 'create' or 'version'")

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True)

if result.returncode != 0:
    raise RuntimeError('Kaggle Dataset upload failed.')

print('Done.')
print(f'Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}')


## Serve The Quantized Chat Model

Start a local NanoChat quantized web server from one exported artifact.


In [ ]:
import time
import requests

SERVE_ARTIFACT_NAME = 'int8_linear'  # choose 'int8_linear', 'fp16_copy', or 'awq_int4'
SERVE_RUNTIME_QUANTIZED = False
SERVER_PORT = 8000
BASE_URL = f'http://127.0.0.1:{SERVER_PORT}'

assert SERVE_ARTIFACT_NAME in QUANT_ARTIFACTS, f'Unknown quant artifact: {SERVE_ARTIFACT_NAME}'
SERVE_QUANT_ARTIFACT = Path(QUANT_ARTIFACTS[SERVE_ARTIFACT_NAME])
assert SERVE_QUANT_ARTIFACT.exists(), f'Missing quant artifact: {SERVE_QUANT_ARTIFACT}'

try:
    if quant_server.poll() is None:
        print('Stopping existing NanoChat quantized server...')
        quant_server.terminate()
        quant_server.wait(timeout=10)
        print('Stopped existing quantized server.')
except NameError:
    pass
except Exception as exc:
    print('Could not stop existing quantized server cleanly:', exc)
    try:
        quant_server.kill()
        quant_server.wait(timeout=10)
        print('Killed existing quantized server.')
    except Exception:
        pass

messages = []

server_env = env.copy()
server_env['NANOCHAT_DISABLE_COMPILE'] = '1'
server_env['TORCH_COMPILE_DISABLE'] = '1'
server_env['OMP_NUM_THREADS'] = '1'

cmd = [
    sys.executable,
    '-m', 'scripts.chat_web_quant',
    '--quant-artifact', str(SERVE_QUANT_ARTIFACT),
    '--num-gpus', '1',
    '--host', '127.0.0.1',
    '--port', str(SERVER_PORT),
    '--max-tokens', '512',
]
if SERVE_RUNTIME_QUANTIZED:
    cmd.append('--runtime-quantized')

print('Starting NanoChat quantized server:')
print(' '.join(cmd))
quant_server = subprocess.Popen(cmd, cwd=WORK_REPO, env=server_env)
print(f'Quantized server process started with PID {quant_server.pid}.')

SERVER_READY = False
for _ in range(60):
    if quant_server.poll() is not None:
        raise RuntimeError(f'NanoChat quantized server exited early with code {quant_server.returncode}')
    try:
        response = requests.get(f'{BASE_URL}/health', timeout=2)
        if response.ok and response.json().get('ready'):
            SERVER_READY = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if SERVER_READY:
    print(f'NanoChat quantized server is ready: {BASE_URL}')
else:
    print(f'NanoChat quantized server is still loading. Wait a bit, then use: {BASE_URL}')


In [ ]:
import json
import requests

BASE = globals().get('BASE_URL', 'http://127.0.0.1:8000')
messages = []

def ask(prompt, temperature=0.8, top_k=50, max_tokens=512):
    messages.append({'role': 'user', 'content': prompt})

    payload = {
        'messages': messages,
        'temperature': temperature,
        'top_k': top_k,
        'max_tokens': max_tokens,
    }

    print('Assistant:', end=' ', flush=True)
    answer = ''

    with requests.post(f'{BASE}/chat/completions', json=payload, stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith('data: '):
                continue

            data = json.loads(line[len('data: '):])
            if data.get('done'):
                break

            token = data.get('token', '')
            answer += token
            print(token, end='', flush=True)

    print()
    messages.append({'role': 'assistant', 'content': answer})
    return answer

print(f'Chatting with {BASE}. Type q, quit, or exit to stop. Type reset to clear history.')
while True:
    prompt = input('\nYou: ')
    command = prompt.strip().lower()
    if command in {'q', 'quit', 'exit'}:
        break
    if command == 'reset':
        messages.clear()
        print('Chat history cleared.')
        continue
    ask(prompt)


In [ ]:
# Stop the NanoChat quantized web server started by the serving cell.
import os
import subprocess
import time

SERVER_PORT = globals().get('SERVER_PORT', 8000)
stopped_any = False

server_process = globals().get('quant_server')
if server_process is not None:
    if server_process.poll() is None:
        print(f'Stopping NanoChat quantized server process {server_process.pid}...')
        server_process.terminate()
        try:
            server_process.wait(timeout=10)
            print('NanoChat quantized server stopped.')
        except subprocess.TimeoutExpired:
            print('Server did not stop after terminate(); killing it...')
            server_process.kill()
            server_process.wait(timeout=10)
            print('NanoChat quantized server killed.')
        stopped_any = True
    else:
        print(f'NanoChat quantized server process already exited with code {server_process.returncode}.')
else:
    print('No notebook `quant_server` variable found.')

try:
    import psutil

    current_pid = os.getpid()
    fallback_processes = []
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        if proc.info['pid'] == current_pid:
            continue
        cmdline = ' '.join(proc.info.get('cmdline') or [])
        if 'scripts.chat_web_quant' not in cmdline:
            continue
        try:
            listening_on_port = any(
                conn.status == psutil.CONN_LISTEN
                and conn.laddr
                and conn.laddr.port == SERVER_PORT
                for conn in proc.net_connections(kind='inet')
            )
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            listening_on_port = False
        if listening_on_port:
            fallback_processes.append(proc)

    for proc in fallback_processes:
        print(f'Stopping fallback chat_web_quant process {proc.pid} on port {SERVER_PORT}...')
        proc.terminate()
    gone, alive = psutil.wait_procs(fallback_processes, timeout=10)
    for proc in alive:
        print(f'Killing fallback chat_web_quant process {proc.pid}...')
        proc.kill()
    if fallback_processes:
        stopped_any = True
except Exception as exc:
    print('Fallback process scan skipped:', exc)

if stopped_any:
    time.sleep(2)
    print('Server shutdown complete.')
else:
    print('No running NanoChat quantized server process found.')
